# First experiments with whole Project architecture setup

In [7]:
%load_ext autoreload
%autoreload 2
%load_ext tensorboard

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [8]:
from constraints.lightning_wrappers.modules import ProjectLightning
from constraints.datatools.datasets import CachedArtificalDataset
from constraints import get_experiment_folder, get_data_folder, show_torch_image
from constraints.transforms.transformers import RigidTransformer
from constraints.computers.loss_computers import ProjectLossComputer
from constraints.losses import OneSideSDFSquare
from constraints.models.affine import ProjectWithTemplateA 
from pathlib import Path

import torch
import pytorch_lightning as pl
FODLER = get_experiment_folder(Path("ex3")/"project_debug")
DATA =get_data_folder() / "artificial" / "downloaded"
TRN_FOLDER = DATA / "trn" / "affine"
VAL_FOLDER = DATA / "val" / "affine"

In [9]:
trn_dataset = CachedArtificalDataset(TRN_FOLDER, sdf_mode="scipy")
val_dataset = CachedArtificalDataset(VAL_FOLDER, sdf_mode="scipy")

In [10]:
from constraints.types import LossInput, LossResult
from torchmetrics.functional.classification import multiclass_jaccard_index    


class CrossEntrAndOneSide(ProjectLossComputer):
    def __init__(self, num_classes=3, weight=1.0):
        super().__init__()
        self.num_classes = num_classes
        self._one_sided = OneSideSDFSquare()
        self._cross_entropy = torch.nn.CrossEntropyLoss()

    @staticmethod
    def _to_labels(x: torch.Tensor) -> torch.Tensor:
        # [B, C, H, W] -> [B, H, W], already-labeled -> long
        if x.ndim == 4:
            return x.argmax(dim=1)
        return x.long()

    def compute(self, loss_input: LossInput) -> LossResult:
        gt_sdf = loss_input.gt_mask_sdf
        gt_mask = loss_input.gt_mask
        pred_mask_logits = loss_input.segmentation_logits
        warped_template = loss_input.warped_template

        loss_seg = self._cross_entropy(pred_mask_logits, gt_mask)
        loss_sdf = self._one_sided(warped_template, gt_sdf)
        loss = loss_seg + loss_sdf

        pred_labels = self._to_labels(pred_mask_logits)
        warped_labels = self._to_labels(warped_template)
        gt_labels = self._to_labels(gt_mask)

        iou_pred_vs_gt = multiclass_jaccard_index(
            preds=pred_labels,
            target=gt_labels,
            num_classes=self.num_classes,
            average="macro",
        )
        iou_warped_vs_gt = multiclass_jaccard_index(
            preds=warped_labels,
            target=gt_labels,
            num_classes=self.num_classes,
            average="macro",
        )

        components = {
            "loss_seg": loss_seg,
            "loss_sdf": loss_sdf,
        }
        logs = {
            "iou/pred_vs_gt": iou_pred_vs_gt,
            "iou/warped_vs_gt": iou_warped_vs_gt,
        }

        return LossResult(total=loss, components=components, logs=logs)

In [11]:
transformer = RigidTransformer()
loss_computer = CrossEntrAndOneSide()
net = ProjectWithTemplateA(max_translation=0.5)
module = ProjectLightning(net, transformer, loss_computer)



config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

In [12]:
from torch.utils.data import DataLoader
from pytorch_lightning.loggers import TensorBoardLogger

BATCH_SIZE = 16
NUM_WORKERS = 4
EPOCHS = 20
LR = 1e-3

trn_loader = DataLoader(
    trn_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

# ProjectLightning in this repo does not define configure_optimizers, so wire it here.
module.configure_optimizers = lambda: torch.optim.Adam(module.parameters(), lr=LR)

tb_logger = TensorBoardLogger(save_dir=str(FODLER), name="tb")

trainer = pl.Trainer(
    max_epochs=EPOCHS,
    accelerator="auto",
    devices="auto",
    logger=tb_logger,
    log_every_n_steps=1,
    enable_checkpointing=False,
)

trainer.fit(module, train_dataloaders=trn_loader, val_dataloaders=val_loader)

TB_LOGDIR = str(FODLER / "tb")
%tensorboard --logdir $TB_LOGDIR

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3060 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/michal/Documents/Skola/diplomka/ctu-constraints/.venv/lib/python3.13/site-packages/pytorch_lightning/utilities/model_summary/model_summary.py:493: The total number of parameters detected may be inaccurate because the model contains an instance of `UninitializedParameter`. To get an accurate number, set `self.example_input_array` in your LightningModule.

  | Name              | Type                 | Params | Mode 
------------------------------------------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/home/michal/Documents/Skola/diplomka/ctu-constraints/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
